# Phan cum sinh vien lop K58KTP - K-Means (K=3)
**Tieu chi:** GPA trung binh tich luy (thang 4)

**Yeu cau:** Dat file `diem.xlsx` cung thu muc voi notebook nay

In [ ]:
# Cap nhat ten file neu can
# !pip install scikit-learn openpyxl pandas matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')
print('Import thanh cong')

## 1. Doc du lieu

In [ ]:
FILE = 'diem.xlsx'

df = pd.read_excel(FILE, sheet_name=0, header=None)

student_ids   = df.iloc[1, 3:].values
student_names = df.iloc[2, 3:].values
grades_raw    = df.iloc[4:, 3:]

print('Tong so sinh vien:', len(student_ids))
print('So mon hoc:', len(grades_raw))

## 2. Lam sach du lieu & tinh GPA

In [ ]:
def to_numeric(val):
    try:
        if pd.isna(val): return np.nan
        s = str(val).strip().lower()
        if s in ['nan', '', '-', '—', '|']: return np.nan
        if hasattr(val, 'year'): return np.nan
        f = float(s)
        return f if 0 <= f <= 4.0 else np.nan
    except:
        return np.nan

grades_clean = grades_raw.apply(lambda col: col.map(to_numeric))
student_avgs = grades_clean.mean(axis=0).values

print('GPA min:', round(float(np.nanmin(student_avgs)), 3))
print('GPA max:', round(float(np.nanmax(student_avgs)), 3))
print('SV co GPA hop le:', np.sum(~np.isnan(student_avgs)), '/', len(student_avgs))

## 3. K-Means Clustering (K=3)

In [ ]:
valid_mask    = ~np.isnan(student_avgs)
avgs_valid    = student_avgs[valid_mask].reshape(-1, 1)
names_valid   = list(student_names[valid_mask])
ids_valid     = list(student_ids[valid_mask])
names_invalid = list(student_names[~valid_mask])
ids_invalid   = list(student_ids[~valid_mask])

K = 3
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = kmeans.fit_predict(avgs_valid)

centers = kmeans.cluster_centers_.flatten()
sorted_idx = np.argsort(centers)[::-1]
rank_map   = {int(sorted_idx[i]): i for i in range(K)}
ranked_labels = np.array([rank_map[int(l)] for l in labels])
centers_sorted = sorted(centers, reverse=True)

print('Tam cum (GPA trung binh):')
for i, c in enumerate(centers_sorted):
    count = int(np.sum(ranked_labels == i))
    print('  Nhom', i+1, ': GPA =', round(c, 3), '|', count, 'sinh vien')

## 4. Hien thi ket qua tung nhom

In [ ]:
def xep_loai(gpa):
    if gpa >= 3.6: return 'Xuat sac'
    elif gpa >= 3.2: return 'Gioi'
    elif gpa >= 2.5: return 'Kha'
    elif gpa >= 2.0: return 'Trung binh'
    return 'Yeu'

group_labels = ['Hoc luc Gioi', 'Hoc luc Kha', 'Hoc luc Trung binh']

all_rows = []
for k in range(K):
    for i in np.where(ranked_labels == k)[0]:
        gpa = float(avgs_valid[i][0])
        all_rows.append({'MSSV': ids_valid[i], 'HoTen': names_valid[i],
                         'GPA': round(gpa, 3), 'Nhom': k+1,
                         'MoTa': group_labels[k], 'XepLoai': xep_loai(gpa)})

for name, sid in zip(names_invalid, ids_invalid):
    all_rows.append({'MSSV': sid, 'HoTen': name, 'GPA': None,
                     'Nhom': None, 'MoTa': 'Thieu du lieu', 'XepLoai': 'N/A'})

result_df = pd.DataFrame(all_rows).sort_values(['Nhom', 'GPA'], ascending=[True, False])
result_df.insert(0, 'STT', range(1, len(result_df)+1))
result_df = result_df.reset_index(drop=True)

SEP = '=' * 65
for k in range(K):
    grp = result_df[result_df['Nhom'] == k+1]
    print('\n' + SEP)
    print('  NHOM', k+1, '-', group_labels[k].upper(),
          '| GPA TB cum:', round(centers_sorted[k], 3), '|', len(grp), 'SV')
    print(SEP)
    print(grp[['STT','MSSV','HoTen','GPA','XepLoai']].to_string(index=False))

if names_invalid:
    print('\n[!]', len(names_invalid), 'SV thieu du lieu:', names_invalid)

## 5. Bieu do phan cum

In [ ]:
colors = ['#2ecc71', '#f39c12', '#e74c3c']
labels_name = ['Nhom 1 - Hoc luc Gioi', 'Nhom 2 - Hoc luc Kha', 'Nhom 3 - Hoc luc TB']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('PHAN CUM SINH VIEN K58KTP - K-MEANS (K=3)', fontsize=14, fontweight='bold', y=1.02)

# --- Bieu do 1: Scatter GPA theo index ---
ax1 = axes[0]
for k in range(K):
    idxs = np.where(ranked_labels == k)[0]
    ax1.scatter(idxs, avgs_valid[idxs], color=colors[k], alpha=0.7, s=50, label=labels_name[k])
    ax1.axhline(centers_sorted[k], color=colors[k], linestyle='--', linewidth=1.5,
                label='TB ' + str(round(centers_sorted[k], 2)))
ax1.set_title('GPA tung sinh vien theo cum', fontweight='bold')
ax1.set_xlabel('Chi so sinh vien')
ax1.set_ylabel('GPA trung binh')
ax1.legend(fontsize=7)
ax1.grid(True, alpha=0.3)

# --- Bieu do 2: Bar chart so luong moi nhom ---
ax2 = axes[1]
counts = [int(np.sum(ranked_labels == k)) for k in range(K)]
bars = ax2.bar(['Nhom 1\n(Gioi)', 'Nhom 2\n(Kha)', 'Nhom 3\n(TB)'], counts,
               color=colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(count) + ' SV', ha='center', fontweight='bold', fontsize=11)
ax2.set_title('So luong sinh vien moi nhom', fontweight='bold')
ax2.set_ylabel('So sinh vien')
ax2.set_ylim(0, max(counts) + 4)
ax2.grid(True, axis='y', alpha=0.3)

# --- Bieu do 3: Histogram phan phoi GPA ---
ax3 = axes[2]
for k in range(K):
    idxs = np.where(ranked_labels == k)[0]
    ax3.hist(avgs_valid[idxs], bins=8, color=colors[k], alpha=0.6,
             label=labels_name[k], edgecolor='white')
ax3.set_title('Phan phoi GPA theo cum', fontweight='bold')
ax3.set_xlabel('GPA trung binh')
ax3.set_ylabel('So sinh vien')
ax3.legend(fontsize=7)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bieudo_phancum.png', dpi=150, bbox_inches='tight')
plt.show()
print('Da luu bieu do: bieudo_phancum.png')

## 6. Xuat file Excel ket qua

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

def border(cell):
    cell.border = Border(left=Side(style='thin'), right=Side(style='thin'),
                         top=Side(style='thin'), bottom=Side(style='thin'))

themes = [
    {'fill': 'C6EFCE', 'font': '276221', 'header': '375623'},
    {'fill': 'FFEB9C', 'font': '9C5700', 'header': '9C5700'},
    {'fill': 'FFC7CE', 'font': '9C0006', 'header': '9C0006'},
]

wb = Workbook()
ws_all = wb.active
ws_all.title = 'Tat ca SV'

ws_all.merge_cells('A1:F1')
ws_all['A1'] = 'DANH SACH PHAN CUM K-MEANS (K=3) - K58KTP'
ws_all['A1'].font = Font(name='Arial', bold=True, size=13, color='FFFFFF')
ws_all['A1'].fill = PatternFill('solid', fgColor='1F497D')
ws_all['A1'].alignment = Alignment(horizontal='center')
ws_all.row_dimensions[1].height = 28

for j, h in enumerate(['STT','MSSV','Ho Ten','GPA TB','Nhom','Xep loai'], 1):
    c = ws_all.cell(row=3, column=j, value=h)
    c.font = Font(name='Arial', bold=True, color='FFFFFF')
    c.fill = PatternFill('solid', fgColor='1F497D')
    c.alignment = Alignment(horizontal='center')
    border(c)

# Dung iterrows() tranh loi ten cot tieng Viet voi itertuples()
for i, (_, row) in enumerate(result_df.iterrows(), 1):
    nhom = row['Nhom']
    t = themes[int(nhom)-1] if pd.notna(nhom) else {'fill': 'F2F2F2', 'font': '666666', 'header': '666666'}
    nhom_str = 'Nhom ' + str(int(nhom)) if pd.notna(nhom) else 'N/A'
    for j, v in enumerate([row['STT'], row['MSSV'], row['HoTen'],
                            row['GPA'], nhom_str, row['XepLoai']], 1):
        c = ws_all.cell(row=3+i, column=j, value=v)
        c.font = Font(name='Arial', size=10, color=t['font'])
        c.fill = PatternFill('solid', fgColor=t['fill'])
        c.alignment = Alignment(horizontal='left' if j==3 else 'center')
        border(c)

for col, w in zip('ABCDEF', [6, 18, 28, 12, 10, 14]):
    ws_all.column_dimensions[col].width = w

for k in range(K):
    t = themes[k]
    ws = wb.create_sheet(title='Nhom ' + str(k+1))
    grp = result_df[result_df['Nhom'] == k+1]

    ws.merge_cells('A1:E1')
    ws['A1'] = 'NHOM ' + str(k+1) + ' - ' + group_labels[k].upper() + ' | GPA TB: ' + str(round(centers_sorted[k],3)) + ' | ' + str(len(grp)) + ' SV'
    ws['A1'].font = Font(name='Arial', bold=True, size=12, color='FFFFFF')
    ws['A1'].fill = PatternFill('solid', fgColor=t['header'])
    ws['A1'].alignment = Alignment(horizontal='center')
    ws.row_dimensions[1].height = 26

    for j, h in enumerate(['STT','MSSV','Ho Ten','GPA TB','Xep loai'], 1):
        c = ws.cell(row=3, column=j, value=h)
        c.font = Font(name='Arial', bold=True, color='FFFFFF')
        c.fill = PatternFill('solid', fgColor=t['header'])
        c.alignment = Alignment(horizontal='center')
        border(c)

    for i, (_, row) in enumerate(grp.iterrows(), 1):
        for j, v in enumerate([i, row['MSSV'], row['HoTen'], row['GPA'], row['XepLoai']], 1):
            c = ws.cell(row=3+i, column=j, value=v)
            c.font = Font(name='Arial', size=10, color=t['font'])
            c.fill = PatternFill('solid', fgColor=t['fill'])
            c.alignment = Alignment(horizontal='left' if j==3 else 'center')
            border(c)

    for col, w in zip('ABCDE', [6, 18, 28, 12, 14]):
        ws.column_dimensions[col].width = w

OUTPUT = 'PhanCum_K58KTP_KMeans.xlsx'
wb.save(OUTPUT)
print('Da luu file:', OUTPUT)
for k in range(K):
    print('  Nhom', k+1, ':', len(result_df[result_df['Nhom']==k+1]), 'sinh vien')